# Sample Transaction DataFrame

A simple Spark DataFrame created from sample transaction data with date, transaction_type, and amount columns.

In [0]:
data = [
    ("01/09/2026", "credite", 20),
    ("02/09/2026", "debit", 40),
]

columns = ["date", "transaction_type", "amount"]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:
from pyspark.sql.functions import *

In [0]:
#reust i want like this sep-2026 -20
df1 = df.withColumn("date", date_format(to_date(col("date"), "dd/MM/yyyy"), "MMM-yyyy"))
df2 = df1.withColumn("new_amount", when(col("transaction_type")== "debit", -col("amount")).otherwise(col("amount")))
#display(df2)

final_df = df2.groupBy("date").agg(sum("new_amount").alias("total_amount"))

display(final_df)

In [0]:
# impelementing word count logic
# Data
data = [ ("hello spark"), ("hello hadoop"), ("spark pyspark"),("spark hi")]
# columns
columns = ["values"]
# creatinf df
df = spark.createDataFrame(data,columns)
df.display()


# now will write the logic
df1 = (df.select(explode(split(col("values"), " ")).alias("word"))
       .groupBy(col("word"))
       .agg(count("*").alias("word_count"))
       .orderBy(col("word_count").desc())
    )
display(df1)
df1.show()

In [0]:
# DataFrame 1 (larger) - employees
data1 = [
    (1, "Alice", "Engineering"),
    (2, "Bob", "Sales"),
    (3, "Charlie", "Engineering"),
    (4, "Diana", "Marketing"),
    (5, "Eve", "Sales"),
]
cols1 = ["emp_id", "name", "dept"]
df1 = spark.createDataFrame(data1, cols1)

# DataFrame 2 (smaller) - department info per employee (good candidate for broadcast)
data2 = [
    (1, "San Francisco", 50000),
    (2, "New York", 40000),
    (3, "San Francisco", 50000),
    (4, "Los Angeles", 45000),
    (5, "New York", 40000),
]
cols2 = ["emp_id", "location", "budget"]
df2 = spark.createDataFrame(data2, cols2)

# Broadcast join: small df2 is broadcast to all executors to avoid shuffle
joined_df = df1.join(broadcast(df2), "emp_id", "inner")

display(joined_df)

In [0]:
# Q Rename Column (name → first_name)
emp_data = [
(1, "Vinay", 50000),
(2, "Raj", 60000),
(3, "Sam", 70000)
]
emp_cols = ["emp_id", "name", "sal"]
emp_df = spark.createDataFrame(emp_data, emp_cols)
emp_df.display()

new_df = emp_df.withColumnRenamed("name", "first_name")
new_df.display()

new_df2 = emp_df.toDF(
"new_emp_id",
"first_name",
"salary"
).show()



# some additional points on the same questions.
"""1. Using withColumnRenamed() (Best Way)
Python
new_df = emp_df.withColumnRenamed("name", "first_name")

✅ Most commonly used
 ✅ Easy to read
 ✅ Interview-friendly

2. Using select() and Alias
Python
from pyspark.sql.functions import col
 
new_df = emp_df.select(
"emp_id",
col("name").alias("first_name"),
"dep_id",
"sal"
)

✅ Useful when selecting only required columns

3. Using toDF()

If you want to rename all columns at once:

Python
new_df = emp_df.toDF(
"emp_id",
"first_name",
"dep_id",
"sal"
)

⚠️ All column names must be supplied in order."""

In [0]:
##3. Second Highest Salary in Each Department
emp_data = [
(1, "Vinay", 10, 90000),
(2, "Raj", 10, 80000),
(3, "Sam", 10, 70000),
(4, "John", 20, 95000),
(5, "Mary", 20, 85000),
(6, "David", 20, 75000),
(7, "Ravi", 30, 60000),
(8, "Tom", 30, 50000)
]

emp_cols = ["emp_id", "name", "dep_id", "sal"]

emp_df = spark.createDataFrame(emp_data, emp_cols)


dep_data = [
(10, "IT"),
(20, "HR"),
(30, "Finance")
]

dep_cols = ["dep_id", "dep_name"]

dep_df = spark.createDataFrame(dep_data, dep_cols)

In [0]:
# Departments With No Employees
emp_data = [
(1, "Vinay", 10, 50000),
(2, "Raj", 20, 60000),
(3, "Sam", 10, 70000)
]

emp_cols = ["emp_id", "name", "dep_id", "sal"]

emp_df = spark.createDataFrame(emp_data, emp_cols)


dep_data = [
(10, "IT"),
(20, "HR"),
(30, "Finance"),
(40, "Admin"),
(50, "Legal")
]

dep_cols = ["dep_id", "dep_name"]

dep_df = spark.createDataFrame(dep_data, dep_cols)

In [0]:
# Join With NULL Practice
emp_data = [
(1, "Vinay", 10),
(2, "Raj", None),
(3, "Sam", 10),
(4, "John", None)
]

emp_cols = ["emp_id", "name", "dep_id"]

emp_df = spark.createDataFrame(emp_data, emp_cols)


dep_data = [
(10, "IT"),
(20, "HR"),
(30, "Finance")
]

dep_cols = ["dep_id", "dep_name"]

dep_df = spark.createDataFrame(dep_data, dep_cols)